# Task 2 — Column Name and Data-Type Standardization
## Women's Clothing: Trends and Analysis in the Fashion Industry

**Pipeline position:** Task 1 cleaned interim dataset → standardized interim dataset.

This notebook continues from Task 1; it does not reload or clean the raw file. It normalizes column names and converts known numeric/boolean fields using safe parsing. It also applies conservative inference to object columns and records conversion failures rather than silently forcing invalid values.

**Input:** `data/interim/womens_clothing_cleaned.csv`  
**Output:** `data/interim/womens_clothing_standardized.csv` and conversion log.

In [1]:
from pathlib import Path
import json
import re
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

INPUT_PATH = ROOT / "data" / "interim" / "womens_clothing_cleaned.csv"
OUTPUT_PATH = ROOT / "data" / "interim" / "womens_clothing_standardized.csv"
LOG_DIR = ROOT / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_PATH.exists():
    raise FileNotFoundError("Run Task_1_Data_Cleaning.ipynb first. Expected: " + str(INPUT_PATH))

df = pd.read_csv(INPUT_PATH, low_memory=False)
print("Loaded Task 1 output:", df.shape)
display(df.head())

Loaded Task 1 output: (1539, 40)


,title,title_orig,price,retail_price,currency_buyer,units_sold,uses_ad_boosts,rating,rating_count,rating_five_count,...,merchant_name,merchant_info_subtitle,merchant_rating_count,merchant_rating,merchant_id,merchant_has_profile_picture,merchant_profile_picture,product_url,product_picture,product_id
0,2020 Summer Vintage Flamingo Print Pajamas Se...,2020 Summer Vintage Flamingo Print Pajamas Se...,16.00,14,EUR,100,0,3.76,54,26.0,...,zgrdejia,(568 notes),568,4.128521,595097d6a26f6e070cb878d1,0,NaN,https://www.wish.com/c/5e9ae51d43d6a96e303acdb0,https://contestimg.wish.com/api/webimage/5e9ae...,5e9ae51d43d6a96e303acdb0
1,SSHOUSE Summer Casual Sleeveless Soirée Party ...,Women's Casual Summer Sleeveless Sexy Mini Dress,8.00,22,EUR,20000,1,3.45,6135,2269.0,...,sarahouse,"83 % avis positifs (17,752 notes)",17752,3.899673,56458aa03a698c35c9050988,0,NaN,https://www.wish.com/c/58940d436a0d3d5da4e95a38,https://contestimg.wish.com/api/webimage/58940...,58940d436a0d3d5da4e95a38
2,2020 Nouvelle Arrivée Femmes Printemps et Été ...,2020 New Arrival Women Spring and Summer Beach...,8.00,43,EUR,100,0,3.57,14,5.0,...,hxt520,86 % avis positifs (295 notes),295,3.989831,5d464a1ffdf7bc44ee933c65,0,NaN,https://www.wish.com/c/5ea10e2c617580260d55310a,https://contestimg.wish.com/api/webimage/5ea10...,5ea10e2c617580260d55310a
3,Hot Summer Cool T-shirt pour les femmes Mode T...,Hot Summer Cool T Shirt for Women Fashion Tops...,8.00,8,EUR,5000,1,4.03,579,295.0,...,allenfan,"(23,832 notes)",23832,4.020435,58cfdefdacb37b556efdff7c,0,NaN,https://www.wish.com/c/5cedf17ad1d44c52c59e4aca,https://contestimg.wish.com/api/webimage/5cedf...,5cedf17ad1d44c52c59e4aca
4,Femmes Shorts d'été à lacets taille élastique ...,Women Summer Shorts Lace Up Elastic Waistband ...,2.72,3,EUR,100,1,3.10,20,6.0,...,happyhorses,"85 % avis positifs (14,482 notes)",14482,4.001588,5ab3b592c3911a095ad5dadb,0,NaN,https://www.wish.com/c/5ebf5819ebac372b070b0e70,https://contestimg.wish.com/api/webimage/5ebf5...,5ebf5819ebac372b070b0e70


## 1. Standardize column names

Convert names to lowercase snake_case, remove punctuation, and resolve any collisions safely.

In [2]:
def standardize_name(name):
    name = str(name).strip()
    name = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", name)
    name = re.sub(r"[^A-Za-z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_").lower()
    return name

original_columns = list(df.columns)
new_columns = []
used = {}
for col in original_columns:
    base = standardize_name(col) or "unnamed_column"
    count = used.get(base, 0)
    used[base] = count + 1
    new_columns.append(base if count == 0 else f"{base}_{count+1}")

rename_map = dict(zip(original_columns, new_columns))
df.columns = new_columns
print("Renamed columns:", rename_map)

Renamed columns: {'title': 'title', 'title_orig': 'title_orig', 'price': 'price', 'retail_price': 'retail_price', 'currency_buyer': 'currency_buyer', 'units_sold': 'units_sold', 'uses_ad_boosts': 'uses_ad_boosts', 'rating': 'rating', 'rating_count': 'rating_count', 'rating_five_count': 'rating_five_count', 'rating_four_count': 'rating_four_count', 'rating_three_count': 'rating_three_count', 'rating_two_count': 'rating_two_count', 'rating_one_count': 'rating_one_count', 'badges_count': 'badges_count', 'badge_local_product': 'badge_local_product', 'badge_product_quality': 'badge_product_quality', 'badge_fast_shipping': 'badge_fast_shipping', 'tags': 'tags', 'product_color': 'product_color', 'product_variation_size_id': 'product_variation_size_id', 'product_variation_inventory': 'product_variation_inventory', 'shipping_option_name': 'shipping_option_name', 'shipping_option_price': 'shipping_option_price', 'shipping_is_express': 'shipping_is_express', 'countries_shipped_to': 'countries_shi

## 2. Convert known numeric and boolean columns safely

Currency symbols and commas are removed from numeric values where possible. Boolean-like text is mapped consistently. Unrecognized values become missing and are counted in the log for review.

In [3]:
numeric_columns = [
    "price", "retail_price", "units_sold", "rating", "rating_count",
    "rating_five_count", "rating_four_count", "rating_three_count",
    "rating_two_count", "rating_one_count", "badges_count",
    "product_variation_inventory", "shipping_option_price",
    "countries_shipped_to", "inventory_total", "merchant_rating_count",
    "merchant_rating"
]
boolean_columns = [
    "uses_ad_boosts", "badge_local_product", "badge_product_quality",
    "badge_fast_shipping", "shipping_is_express", "has_urgency_banner",
    "merchant_has_profile_picture"
]

conversion_log = {"numeric": {}, "boolean": {}, "inferred_numeric": {}, "column_rename_map": rename_map}

for col in numeric_columns:
    if col in df.columns:
        before_non_null = int(df[col].notna().sum())
        cleaned = (df[col].astype("string")
                   .str.replace(",", "", regex=False)
                   .str.replace(r"[^0-9eE+\-.]", "", regex=True)
                   .replace("", pd.NA))
        converted = pd.to_numeric(cleaned, errors="coerce")
        conversion_log["numeric"][col] = {
            "non_null_before": before_non_null,
            "conversion_to_missing": int(before_non_null - converted.notna().sum())
        }
        df[col] = converted

true_values = {"true", "yes", "1", "y", "t"}
false_values = {"false", "no", "0", "n", "f"}

for col in boolean_columns:
    if col in df.columns:
        original = df[col]
        normalized = original.astype("string").str.strip().str.lower()
        mapped = normalized.map(lambda x: True if x in true_values else (
            False if x in false_values else pd.NA
        ))
        conversion_log["boolean"][col] = {
            "unrecognized_non_null": int((original.notna() & mapped.isna()).sum())
        }
        df[col] = mapped.astype("boolean")

display(df.dtypes.to_frame("standardized_dtype"))

,standardized_dtype
title,str
title_orig,str
price,Float64
retail_price,Int64
currency_buyer,str
units_sold,Int64
uses_ad_boosts,boolean
rating,Float64
rating_count,Int64
rating_five_count,Float64


## 3. Conservative dynamic type detection

For remaining object/string columns, infer numeric only when at least 95% of non-empty values can be parsed as numbers. Otherwise retain the column as text/category. This prevents IDs, URLs, tags, and product names from being incorrectly converted.

In [4]:
for col in df.select_dtypes(include=["object", "string"]).columns:
    values = df[col].dropna().astype(str).str.strip()
    if values.empty:
        continue
    numeric_candidate = pd.to_numeric(
        values.str.replace(",", "", regex=False), errors="coerce"
    )
    parse_ratio = float(numeric_candidate.notna().mean())
    if parse_ratio >= 0.95:
        before_non_null = int(df[col].notna().sum())
        df[col] = pd.to_numeric(
            df[col].astype("string").str.replace(",", "", regex=False),
            errors="coerce"
        )
        conversion_log["inferred_numeric"][col] = {
            "parse_ratio": round(parse_ratio, 4),
            "conversion_to_missing": int(before_non_null - df[col].notna().sum())
        }

# crawl_month is a time-period label in many versions of this dataset.
# Keep it as text unless its actual values are confirmed to be parseable dates.
if "crawl_month" in df.columns:
    df["crawl_month"] = df["crawl_month"].astype("string")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)

with open(LOG_DIR / "task2_conversion_log.json", "w", encoding="utf-8") as f:
    json.dump(conversion_log, f, indent=4, default=str)

print("Task 2 complete. Saved:", OUTPUT_PATH)
print("Shape:", df.shape)
display(df.dtypes.to_frame("final_dtype"))

Task 2 complete. Saved: d:\OneDrive\Desktop\Brainybeams\Womens_Clothing_ML_Project\data\interim\womens_clothing_standardized.csv
Shape: (1539, 40)


,final_dtype
title,str
title_orig,str
price,Float64
retail_price,Int64
currency_buyer,str
units_sold,Int64
uses_ad_boosts,boolean
rating,Float64
rating_count,Int64
rating_five_count,Float64


## 4. Validate the standardized dataset

Check duplicate column names, type consistency, and conversion side effects.

In [5]:
assert df.columns.is_unique, "Duplicate column names remain."
validation = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
})
display(validation)
print("Standardized dataset saved for Task 3.")

,dtype,missing_count,missing_percent
title,str,0,0.00
title_orig,str,0,0.00
price,Float64,0,0.00
retail_price,Int64,0,0.00
currency_buyer,str,0,0.00
units_sold,Int64,0,0.00
uses_ad_boosts,boolean,0,0.00
rating,Float64,0,0.00
rating_count,Int64,0,0.00
rating_five_count,Float64,43,2.79


Standardized dataset saved for Task 3.


### Task 2 outcome
- Column names are consistent snake_case.
- Known numeric and boolean fields are converted safely.
- Remaining columns are dynamically inferred only when confidence is high.
- Conversion failures and rename mappings are logged.
- The standardized dataset is passed to Task 3.